> 📖 **參考解答**：請先在學生版自己試，卡住再對照這份。學生版在 [`notebooks/`](../notebooks/)，這份與學生版的差別只在練習題的空格（`____`）已填好。

# PBAIH12-A 實作一：認識 MS/MS 與 MS1 線索（參考解答）

**目標**：上一單元我們得到一張可以做統計的特徵表，也找到了可能很重要的 feature，但**不知道它們是誰**。這份實作分兩步：先看「只有 m/z 和同位素」能讓我們猜到什麼，再學會怎麼**比較兩張 MS/MS 譜**。

**資料**：MassBank 2026.11 的學生子集：只含授權為 **CC BY 或 CC0**、正離子 [M+H]+、標準品確認的 MS/MS 譜（約 4.9 萬筆、約 4,600 個化合物、24 個貢獻者的實驗室）。每筆紀錄都帶有作者與授權；轉載請標示 MassBank 與作者。來源：Zenodo doi:10.5281/zenodo.24082491。

| 部分 | 內容 |
|---|---|
| Part 0 | 環境與資料（**上課一開始先執行**） |
| Part 1 | 看一筆 MassBank 紀錄、加成物分布（練習 1） |
| Part 2 | 一個 m/z 有幾個候選？調 tolerance 看（練習 2） |
| Part 3 | 理論同位素分布（練習 3） |
| Part 4 | 手算 cosine：兩張玩具譜（練習 4） |
| Part 5 | 用 cosine 比較真實的譜：同一化合物、不同實驗室（練習 5） |
| 延伸 E1 | 同位素比例相似度（**課後／有餘力**） |

> 這裡的 **Part** 是 notebook 的章節；「第 1 節」「第 2 節」指上課的第幾節。

每個 Part 都分成：**為什麼** → **怎麼做** → **看結果** → **想一想**。練習題是「補完這一行」，空格寫成 `____`。

🆘 **卡住時**：① 先讀錯誤訊息的最後一行（通常會說是哪個變數或哪一行）→ ② 問 Colab 內建的 Gemini（把錯誤訊息貼給它，請它「用中文解釋」）→ ③ 打開下方的 💡 提示 → ④ 舉手問老師或助教。

## 執行環境

本 notebook 以 Google Colab 為準（Python 3.13）。需要的套件：

| 套件 | 版本 | 備註 |
|---|---|---|
| matchms | 1.0.0 | Colab 沒有內建，Part 0-a 會安裝；用來讀譜與計算相似度 |
| pandas、numpy、matplotlib | Colab 內建 | matchms 安裝時可能會把它們升級，所以安裝後有時需要重新啟動 |

圖表的文字用英文，因為 Colab 預設沒有中文字型。

---
# 核心路徑（Part 0–2 在上課第 1 節後段，Part 3–5 在第 2 節前段）

## Part 0. 環境與資料（上課一開始先執行）

**為什麼**：安裝套件和下載資料需要等一下，請在上課一開始就先執行，趁它在跑的時候聽老師開場。

**怎麼做**：依序執行 0-a（安裝）和 0-b（載入套件、下載並讀入資料）。若 0-a 要你重新啟動工作階段，照做後**直接執行 0-b**。

> **`.parquet` 是什麼？** 0-b 下載的資料是 `.parquet` 檔：一種存表格的檔案格式，用 `pd.read_parquet` 讀進 pandas 之後，和 CSV 一樣是 DataFrame。它比 CSV 小、讀得快，而且一格裡可以放一串數字，所以每張譜的 m/z 和強度各存成一欄（一格就是這張譜的所有峰）。Excel 不能直接開它，在 Colab 用 pandas 讀就好；左側檔案面板點不開是正常的。

> 開發／本機執行：若已有資料，可先設定環境變數 `PBAIH12_DATA_DIR` 指向放 parquet 檔的資料夾，0-b 就會略過下載。

In [ ]:
# 0-a 安裝 matchms（只在 Colab 上需要；上課一開始先執行這一格）
import sys
import importlib.metadata as md

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q matchms==1.0.0")

# 安裝可能會升級 numpy／pandas／matplotlib；如果這些套件在安裝前已經被載入，記憶體裡還是舊版，要重新啟動才會換新
stale = [p for p in ("numpy", "pandas", "matplotlib") if p in sys.modules and md.version(p) != sys.modules[p].__version__]
if stale:
    print("⚠️ 這些套件已升級，但記憶體裡還是舊版：", stale)
    print("請按上方選單「執行階段 → 重新啟動工作階段」，然後【直接執行 0-b】（不用再執行 0-a）。")
else:
    print("安裝完成，請執行 0-b。")

In [ ]:
# 0-b 載入套件、取得資料、讀入譜庫子集
import os, sys, glob, zipfile, logging
import importlib.metadata as md

IN_COLAB = "google.colab" in sys.modules
stale = [p for p in ("numpy", "pandas", "matplotlib") if p in sys.modules and md.version(p) != sys.modules[p].__version__]
if stale:
    raise RuntimeError(f"{stale} 的版本在安裝後改變了。請按「執行階段 → 重新啟動工作階段」，再重新執行這一格。")

# 取得資料：優先順序 = 環境變數 PBAIH12_DATA_DIR → 已下載（或手動放好）的資料夾 → 從課程 GitHub 下載
DATA_URL = "https://github.com/wang3yuan/PBAIH/releases/download/data-v2/PBAIH12_data.zip"
DATA_ROOT = "pbaih12_data"

def find_data_dir(*roots):
    """在這些資料夾底下找含有 massbank_pos_ms2.parquet 的資料夾。"""
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, "**", "massbank_pos_ms2.parquet"), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    return None

DATA_DIR = os.environ.get("PBAIH12_DATA_DIR") or find_data_dir(DATA_ROOT, ".")
if DATA_DIR is None:
    import urllib.request
    os.makedirs(DATA_ROOT, exist_ok=True)
    zip_path = os.path.join(DATA_ROOT, "PBAIH12_data.zip")
    print("下載中（約 10 MB）...")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_ROOT)
    DATA_DIR = find_data_dir(DATA_ROOT)
assert DATA_DIR is not None, "找不到資料，請看 Part 0.1"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message=".*hdf5plugin.*")   # matchms 內部套件的無害提示
import matchms
from matchms import Spectrum
from matchms.similarity import CosineGreedy
logging.getLogger("matchms").setLevel(logging.ERROR)   # 關掉 matchms 的警告訊息

for name, mod in [("matchms", matchms), ("numpy", np), ("pandas", pd), ("matplotlib", matplotlib)]:
    print(f"{name:11s} {mod.__version__}")
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| 本機")

lib = pd.read_parquet(os.path.join(DATA_DIR, "massbank_pos_ms2.parquet"))
meta_ms2 = pd.read_parquet(os.path.join(DATA_DIR, "massbank_ms2_meta.parquet"))
print(f"譜庫子集：{len(lib):,} 筆 MS/MS、{lib.ik14.nunique():,} 個化合物、{lib.contributor.nunique()} 個貢獻者")
print("授權：", lib.license.value_counts().to_dict())


# 前處理：把一筆紀錄變成 matchms 的 Spectrum。兩個步驟都寫在這裡，不交給套件預設去做：
#   (1) 先移除前驅物附近的峰（m/z ≥ 前驅物 m/z − 1.6）：沒有碎裂的前驅物離子沒有結構資訊，而且常常是最大的峰
#   (2) 再去掉低於（剩下的）基峰 1% 的峰，當作雜訊
PREC_OFFSET = -1.6
def get_spec(r, min_rel=0.01):
    mz = np.asarray(r.mz, dtype=float); it = np.asarray(r.intensity, dtype=float)
    keep = mz < r.precursor_mz + PREC_OFFSET
    mz, it = mz[keep], it[keep]
    if len(mz):
        keep = it >= min_rel * it.max()
        mz, it = mz[keep], it[keep]
    return Spectrum(mz=mz, intensities=it, metadata={"precursor_mz": float(r.precursor_mz)})

### Part 0.1 如果安裝或下載失敗怎麼辦

| 狀況 | 怎麼辦 |
|---|---|
| 0-a 印出「這些套件已升級，但記憶體裡還是舊版」 | 按「執行階段 → 重新啟動工作階段」，然後**直接執行 0-b**（不用再執行 0-a）。 |
| 0-a 跑完出現紅字的 `ERROR: pip's dependency resolver ...`，提到 `pandas`、`requests` 或 `google-colab` | 通常可以忽略；只要 0-b 能順利跑完就好。 |
| `pip` 顯示 `No matching distribution found for matchms==1.0.0` | 套件版本和 Python 不合。執行 `!python --version`，把結果告訴老師或助教。 |
| 0-b 顯示 `RuntimeError ... 請按「執行階段 → 重新啟動工作階段」` | 照訊息做：重新啟動後再執行 0-b。 |
| 下載時顯示 `HTTP Error`（例如 404、429）或連線失敗 | 用瀏覽器打開 https://github.com/wang3yuan/PBAIH/releases/tag/data-v2 ，手動下載 `PBAIH12_data.zip`，在 Colab 左側「檔案」面板上傳，再執行下面「手動放資料」那段。 |
| 網路逾時、下載到一半中斷 | 刪掉 `pbaih12_data` 資料夾後再執行一次 0-b。 |

手動放資料（上傳 zip 之後）：

```python
import zipfile
zipfile.ZipFile("PBAIH12_data.zip").extractall("pbaih12_data")
```
然後重新執行 0-b。

> **每一份 notebook 都要各自執行一次 Part 0**（Colab 的每份 notebook 是獨立的執行環境，不會共用已安裝的套件和下載的資料）。

## Part 1. 看一筆 MassBank 紀錄

**為什麼**：在做任何比對之前，先看一眼資料：一筆 MS/MS 紀錄裡到底有什麼？同一個化合物會有幾筆？不同實驗室的儀器和條件一樣嗎？

**怎麼做**：每一列是**一張 MS/MS 譜**。`mz`、`intensity` 是這張譜的碎片峰（強度已正規化成最高峰＝1）。`ik14` 是 InChIKey 的前 14 碼，代表分子骨架（不含立體化學），我們用它當「是不是同一個化合物」的答案。`confidence` 是這筆**參考譜本身**的鑑定信心（用標準品測的），不是你拿樣品去比對之後的等級。

儀器欄位的縮寫：`QTOF` 是四極柱＋飛行時間；`QFT`、`ITFT` 是含 Orbitrap（傅立葉轉換）的機型。

In [ ]:
# 1-1 看一筆紀錄的欄位（峰列表另外顯示）
row = lib[lib.name.str.lower() == "caffeine"].iloc[0]
print(row.drop(["mz", "intensity"]).to_string())
print()
print(f"這張譜有 {len(row.mz)} 個碎片峰；前 5 個 (m/z, 相對強度)：")
for m, i in list(zip(row.mz, row.intensity))[:5]:
    print(f"  {m:10.4f}  {i:.3f}")

In [ ]:
# 1-2 同一個化合物有幾筆譜、來自幾個實驗室？
caf = lib[lib.ik14 == row.ik14]
print(f"咖啡因：{len(caf)} 筆譜、{caf.contributor.nunique()} 個貢獻者")
display(caf.groupby(["contributor", "instrument_type"]).size().rename("n_spectra").reset_index())
print("碰撞能量的種類（同一化合物在不同條件下的譜不一樣）：")
print(caf.collision_energy.value_counts().head(8).to_string())

### 練習 1：加成物分布

學生子集只留下 [M+H]+，但 `meta_ms2` 是**全部離子模式**的 MS/MS 紀錄的欄位資訊（不含峰；同樣只含 CC BY 與 CC0 紀錄）。請數一數 `precursor_type`（前驅物的加成物種類）各有幾筆，包含沒填的。

In [ ]:
# 練習 1：加成物（precursor_type）出現次數，包含缺值
adduct_counts = meta_ms2["precursor_type"].value_counts(dropna=False)
display(adduct_counts.head(8))
print(f"全部 MS/MS 紀錄 {len(meta_ms2):,} 筆；[M+H]+ 占 {adduct_counts['[M+H]+'] / len(meta_ms2):.0%}")

<details><summary>💡 練習 1 提示（先自己試，卡住再打開）</summary>

`pandas` 的 `Series.value_counts()` 會統計每個值出現幾次；缺值要用參數 `dropna=False` 才會被算進去。

<details><summary>參考答案</summary>

```python
adduct_counts = meta_ms2["precursor_type"].value_counts(dropna=False)
```
</details>
</details>

**想一想**
1. 為什麼同一個化合物會有這麼多筆不同的譜？（提示：看上面的碰撞能量與儀器）
2. 如果你的 feature 是在**負離子模式**測到的，這個學生子集能用嗎？你還需要什麼？
3. 同一個代謝物可能以 [M+H]+、[M+Na]+、[M+NH4]+ 出現，如果你不先決定加成物，後面算出的「分子量」會發生什麼事？

## Part 2. 一個 m/z 有幾個候選？

**為什麼**：手上只有一個 feature 的 m/z 時，最直接的做法是「去資料庫找質量一樣的化合物」。問題是：**質量一樣的化合物不只一個**。tolerance 設多寬，候選就多多少？

**怎麼做**：先把譜庫整理成「每個化合物一列」（同一化合物的前驅物 m/z 取中位數），再寫一個函式：給一個 m/z 和 tolerance，回傳落在範圍內的化合物。

**ppm 怎麼算**：ppm＝百萬分之一，是**相對**誤差。m/z 300 的 ±10 ppm 是 300 × 10 × 10⁻⁶ ＝ ±0.003。

In [ ]:
# 2-1 每個化合物一列
comp = (lib.groupby("ik14")
        .agg(name=("name", "first"), formula=("formula", "first"),
             mz=("precursor_mz", "median"), n_spectra=("accession", "size"))
        .sort_values("mz").reset_index())
print(len(comp), "個化合物；前驅物 m/z 範圍", round(comp.mz.min(), 1), "–", round(comp.mz.max(), 1))
comp.head()

### 練習 2：寫一個「找候選」的函式

請補完 `hi` 這一行（上界），讓函式回傳 m/z 在 `mz ± ppm` 範圍內的化合物。

In [ ]:
def candidates(mz, ppm):
    """前驅物 m/z 落在 mz ± ppm 範圍內的化合物。"""
    lo = mz * (1 - ppm * 1e-6)
    hi = mz * (1 + ppm * 1e-6)
    return comp[(comp.mz >= lo) & (comp.mz <= hi)]

obs_mz = 253.0972           # 假設這是你的 feature（[M+H]+ 的 m/z）
cand = candidates(obs_mz, 10).copy()
cand["error_ppm"] = (cand.mz - obs_mz) / obs_mz * 1e6
display(cand[["name", "formula", "mz", "error_ppm"]].round(3))
for ppm in [10, 50, 200, 1000]:
    print(f"± {ppm:>4} ppm 內的候選：{len(candidates(obs_mz, ppm))} 個")

<details><summary>💡 練習 2 提示（先自己試，卡住再打開）</summary>

下界是 `mz * (1 - ppm * 1e-6)`，上界只是把減號換成加號。

<details><summary>參考答案</summary>

```python
hi = mz * (1 + ppm * 1e-6)
```
</details>
</details>

**看結果**：±10 ppm 內有幾個候選？其中有幾個**分子式完全一樣**？質量一樣、分子式也一樣的候選，只靠質量就完全分不開，這就是**同分異構物**。哪一個候選的分子式和別人不同？它的差別在哪裡？（Part 3 的同位素可以幫忙。）

接下來看整體情況：隨機挑 1,000 個化合物，把它們的 m/z 當成「觀察到的 feature」，算庫裡還有幾個**其他**化合物落在同一個範圍。

In [ ]:
# 2-2 候選數 vs tolerance（隨機 1,000 個化合物；每個都扣掉自己）
rng = np.random.default_rng(0)
sample = rng.choice(comp.mz.values, 1000, replace=False)
rows = []
for ppm in [1, 2, 5, 10, 20, 50]:
    n_other = np.array([len(candidates(x, ppm)) - 1 for x in sample])
    rows.append({"ppm": ppm, "mean_others": n_other.mean(), "share_with_other": (n_other >= 1).mean()})
tol_table = pd.DataFrame(rows)
display(tol_table.round(2))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(tol_table.ppm, tol_table.share_with_other * 100, "o-")
ax.set_xscale("log"); ax.set_xlabel("m/z tolerance (± ppm)")
ax.set_ylabel("% compounds with ≥1 other candidate")
ax.set_title(f"Same-m/z candidates in a {len(comp):,}-compound library")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 2-3 如果質量精度只有「單位解析」（±0.5 Da）呢？
def candidates_da(mz, da):
    return comp[(comp.mz >= mz - da) & (comp.mz <= mz + da)]
for da in [0.01, 0.05, 0.5]:
    n_other = np.array([len(candidates_da(x, da)) - 1 for x in sample])
    print(f"±{da} Da：平均 {n_other.mean():.1f} 個其他候選，中位數 {np.median(n_other):.0f}")

**想一想**
1. 這個譜庫只有約 4,600 個化合物。真實世界可能的代謝物與藥物、環境物質多得多，候選會更多還是更少？
2. 把 tolerance 從 ±10 ppm 縮到 ±1 ppm，候選數少了多少？如果質量精度再好一倍，能保證只剩一個候選嗎？為什麼？
3. 同分子式的候選（同分異構物）要靠什麼資訊才分得開？

## Part 3. 理論同位素分布

**為什麼**：除了 m/z，feature 旁邊還有 **M+1、M+2…** 同位素峰，它們的相對強度反映分子裡有幾個 C，以及有沒有 S、Cl、Br。這是 MS1 層級另一條線索。

**怎麼做**：下面的函式把一個分子式的各元素同位素天然豐度「卷積」起來，得到 M、M+1、M+2、M+3 的相對強度（M 設為 100）。
常見元素的同位素天然豐度取到三、四位小數，是教學用的數值。

| 元素 | 主要同位素與豐度 | 次要同位素 |
|---|---|---|
| C | ¹²C 98.93% | ¹³C 1.07% |
| H | ¹H 99.9885% | ²H 0.0115% |
| N | ¹⁴N 99.636% | ¹⁵N 0.364% |
| O | ¹⁶O 99.757% | ¹⁷O 0.038%、¹⁸O 0.205% |
| S | ³²S 94.99% | ³³S 0.75%、³⁴S 4.25% |
| Cl | ³⁵Cl 75.76% | ³⁷Cl 24.24% |
| Br | ⁷⁹Br 50.69% | ⁸¹Br 49.31% |

In [ ]:
import re

# 各元素同位素：(與最輕同位素的整數質量差, 天然豐度)
ISO = {"C": [(0, .9893), (1, .0107)], "H": [(0, .999885), (1, .000115)],
       "N": [(0, .99636), (1, .00364)], "O": [(0, .99757), (1, .00038), (2, .00205)],
       "S": [(0, .9499), (1, .0075), (2, .0425), (4, .0001)],
       "Cl": [(0, .7576), (2, .2424)], "Br": [(0, .5069), (2, .4931)],
       "P": [(0, 1.0)], "F": [(0, 1.0)]}

def isotope_pattern(formula, n=5):
    """回傳 M, M+1, ... M+(n-1) 的相對強度（M = 100）。只支援 ISO 表裡的元素。"""
    atoms = {el: int(k or 1) for el, k in re.findall(r"([A-Z][a-z]?)(\d*)", formula)}
    missing = set(atoms) - set(ISO)
    if missing:
        raise ValueError(f"ISO 表裡還沒有這些元素：{sorted(missing)}（可以自己加進去）")
    dist = np.zeros(n); dist[0] = 1.0
    for el, k in atoms.items():
        base = np.zeros(n)
        for shift, ab in ISO[el]:
            if shift < n:
                base[shift] += ab
        for _ in range(k):
            dist = np.convolve(dist, base)[:n]
    return dist / dist[0] * 100

formulas = {"Caffeine": "C8H10N4O2", "Metoclopramide": "C14H22ClN3O2", "Sulfisoxazole": "C11H13N3O3S"}
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)
for ax, (nm, f) in zip(axes, formulas.items()):
    p = isotope_pattern(f)
    ax.bar(range(len(p)), p, color="#0F8B8D")
    ax.set_xticks(range(len(p))); ax.set_xticklabels(["M", "M+1", "M+2", "M+3", "M+4"])
    ax.set_title(f"{nm}\n{f}", fontsize=10)
    for i, v in enumerate(p[1:4], 1):
        ax.text(i, v + 2, f"{v:.1f}", ha="center", fontsize=9)
axes[0].set_ylabel("relative intensity (M = 100)")
plt.tight_layout(); plt.show()

### 練習 3：用 M+2／M 看出含 Cl 或 S 的化合物

請算出每個分子式的 **M+2 對 M 的強度比（%）**。`isotope_pattern` 回傳的陣列 `pat` 中，`pat[0]` 是 M，`pat[2]` 是 M+2。

In [ ]:
for nm, f in formulas.items():
    pat = isotope_pattern(f)
    ratio_m2 = pat[2] / pat[0] * 100
    print(f"{nm:15s} {f:14s} M+1 = {pat[1]:5.1f}%   M+2 = {ratio_m2:5.1f}%")

<details><summary>💡 練習 3 提示（先自己試，卡住再打開）</summary>

`pat[0]` 是 M 的強度、`pat[2]` 是 M+2 的強度；比值乘 100 就是百分比。

<details><summary>參考答案</summary>

```python
ratio_m2 = pat[2] / pat[0] * 100
```
</details>
</details>

**想一想**
1. 哪一個分子的 M+2 特別高？它含有什麼元素？（Cl 的 ³⁷Cl 約占 ³⁵Cl 的三分之一。含一個 S 的分子 M+2 有多高？）
2. 咖啡因（8 個 C、4 個 N）的 M+1 約 10%；如果一個分子的 M+1 是 20%，大約有幾個 C？（每個 C 約貢獻 1.1%，N、H、O 也會貢獻一點：20 ÷ 1.1 ≈ 18，再扣掉其他元素的貢獻。）
3. 同位素分布能不能「單獨」決定分子式？為什麼質量精度和同位素通常要一起用？

## Part 4. 手算 cosine：兩張玩具譜

**為什麼**：相似度不是黑盒子。要判斷兩張 MS/MS 譜像不像，先把兩張譜的碎片**對齊**：m/z 相同（差在 tolerance 內）的碎片放在同一個位置，各自排成向量，沒有的記 0。**cosine ＝ 兩個向量的內積 ÷ 兩個向量長度的乘積**。自己算一次，才知道分數為什麼高、為什麼低。

**怎麼做**：下面兩張「玩具譜」各有 4 個峰（和講課裡手算的是同一組）。m/z 77、91、119 兩邊都有（對得上），105 只有 A 有，133 只有 B 有。我們把五個 m/z 位置依序排成向量。

In [ ]:
# 4-1 玩具譜：A、B 各 4 個峰（前驅物設 150，比所有碎片都大）
toy_A = Spectrum(mz=np.array([77.0, 91.0, 105.0, 119.0]), intensities=np.array([10.0, 5.0, 2.0, 1.0]), metadata={"precursor_mz": 150.0})
toy_B = Spectrum(mz=np.array([77.0, 91.0, 119.0, 133.0]), intensities=np.array([10.0, 4.0, 1.0, 3.0]), metadata={"precursor_mz": 150.0})

#         m/z： 77  91  105  119  133
a = np.array([10,  5,   2,   1,   0], dtype=float)
b = np.array([10,  4,   0,   1,   3], dtype=float)
print("A =", a); print("B =", b)

**先想想看**：兩張譜有三個峰對得上，其中 m/z 77 都是最大的峰，你猜 cosine 會接近 0 還是接近 1？寫下你的猜測，再往下算。

### 練習 4：補完 cosine 的算式

In [ ]:
dot = np.sum(a * b)                                   # 內積：對應位置相乘再加總
cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))   # 除以兩個向量的長度（範數）
print(f"內積 = {dot:.0f}，|A| = {np.linalg.norm(a):.2f}，|B| = {np.linalg.norm(b):.2f}")
print(f"手算 cosine = {cos:.4f}")
print(f"（MS-DIAL 的 dot product 是 cosine 的平方：{cos**2:.4f}）")

# 對照 matchms（remove_precursor=False：前驅物附近的峰我們在 Part 5 自己處理，這裡的玩具譜不需要）
res = CosineGreedy(tolerance=0.01, remove_precursor=False).pair(toy_A, toy_B)
print(f"matchms：cosine = {float(res['score']):.4f}，對上的峰數 = {int(res['matches'])}")

<details><summary>💡 練習 4 提示（先自己試，卡住再打開）</summary>

內積是兩個向量對應位置相乘再加總：`np.sum(a * b)`。長度用 `np.linalg.norm(...)`，cosine = 內積 ÷（|A|×|B|）。

<details><summary>參考答案</summary>

```python
dot = np.sum(a * b)
cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))
```
</details>
</details>

In [ ]:
# 4-2 如果兩張譜只對上 1 個峰，分數會怎樣？（只留 m/z 77 的大峰）
a1 = np.array([0, 100, 0, 0, 0], dtype=float)
b1 = np.array([0, 100, 0, 0, 0], dtype=float)
print("兩張譜都只有同一個峰：cosine =", a1 @ b1 / (np.linalg.norm(a1) * np.linalg.norm(b1)))

# 4-3 強度開根號（intensity_power = 0.5）：小峰的影響變大
cos_sqrt = (a ** 0.5) @ (b ** 0.5) / (np.linalg.norm(a ** 0.5) * np.linalg.norm(b ** 0.5))
res2 = CosineGreedy(tolerance=0.01, intensity_power=0.5, remove_precursor=False).pair(toy_A, toy_B)
print(f"強度開根號後：手算 {cos_sqrt:.4f}，matchms {float(res2['score']):.4f}")

**想一想**
1. 兩張譜如果**只有一個峰**，而且剛好都是最大的峰，cosine 是多少？這種「1.000」可不可信？（下一份實作會看到真實的例子。）
2. 把強度開根號之後，分數變高還是變低？小峰的影響變大還是變小？
3. 未對上的峰（A 的 105、B 的 133）有沒有被算進分數？在哪裡算進去的？

## Part 5. 用 cosine 比較真實的譜

**為什麼**：MS/MS 是**碎片的指紋**：前驅物離子被打碎後，每個碎片的 m/z 和強度合起來是這個結構的特徵。要判斷兩張譜是不是同一個化合物，最直觀的方法是**鏡像圖**：上面畫查詢譜、下面畫譜庫譜，m/z 對得上的峰標成珊瑚色，沒對上的是灰色。

**怎麼做**：先看**前處理**。`get_spec`（在 Part 0-b 定義）做了兩件事：① 先移除**前驅物附近的峰**（m/z ≥ 前驅物 − 1.6）：沒有碎裂的前驅物離子常常是最大的峰，但它沒有結構資訊；② 再去掉低於基峰 1% 的峰，當作雜訊。**鏡像圖畫的就是前處理之後留下的峰**。（matchms 1.0 的 `CosineGreedy` 預設也會做這兩件事；我們自己寫出來，並傳 `remove_precursor=False`，這樣畫出來的和算分數用的是同一批峰。）

下面先畫咖啡因的一張譜，再做練習 5。

In [ ]:
def plot_spectrum(spec, title="", ax=None, color="#0F8B8D"):
    ax = ax or plt.subplots(figsize=(6.5, 3.4))[1]
    ax.vlines(spec.peaks.mz, 0, spec.peaks.intensities, color=color, lw=1.5)
    ax.set_ylim(0, 1.1); ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity"); ax.set_title(title, fontsize=10)
    return ax

def plot_mirror(q, l, tol=0.025, title=""):
    """上：查詢譜 q；下：譜庫譜 l；m/z 對得上的峰標成珊瑚色，沒對上的灰色。只畫前處理後留下的峰。"""
    fig, ax = plt.subplots(figsize=(7, 3.8))
    lm, li = l.peaks.mz, l.peaks.intensities
    qm, qi = q.peaks.mz, q.peaks.intensities
    for m, i in zip(qm, qi):
        ax.vlines(m, 0, i, color="#EE6C4D" if np.any(np.abs(lm - m) <= tol) else "#8D99AE", lw=1.5)
    for m, i in zip(lm, li):
        ax.vlines(m, 0, -i, color="#EE6C4D" if np.any(np.abs(qm - m) <= tol) else "#8D99AE", lw=1.5)
    ax.axhline(0, color="k", lw=0.6); ax.set_ylim(-1.1, 1.1)
    ax.plot([], [], color="#EE6C4D", lw=2, label="matched"); ax.plot([], [], color="#8D99AE", lw=2, label="not matched")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity"); ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()

spec_a = get_spec(row)
plot_spectrum(spec_a, f"{row['name']}  [M+H]+ {row.precursor_mz:.4f}  CE {row.collision_energy}  ({row.contributor})")
plt.tight_layout(); plt.show()
print(f"原本 {len(row.mz)} 個峰，前處理後留下 {len(spec_a.peaks.mz)} 個（前驅物附近的峰已移除）")

In [ ]:
sim = CosineGreedy(tolerance=0.025, remove_precursor=False)    # 碎片 tolerance 0.025 Da

### 練習 5：同一個化合物、不同實驗室

請從咖啡因的紀錄中，挑出**不同貢獻者**的所有譜（和 Part 1 取出的那一筆 `row` 比），算它們和上面那張譜的 cosine，看看分數的分布；再把最像的那一筆疊成鏡像圖。

In [ ]:
caf = lib[lib.ik14 == row.ik14]
others = caf[caf.contributor != row.contributor]            # 不同貢獻者的咖啡因譜
others = others.assign(cosine=[float(sim.pair(spec_a, get_spec(r))["score"]) for _, r in others.iterrows()])
ranked = others[["contributor", "instrument_type", "collision_energy", "cosine"]].sort_values("cosine", ascending=False)
print("分數最高的 5 筆與最低的 5 筆：")
display(pd.concat([ranked.head(5), ranked.tail(5)]).round(3))
print(f"{len(others)} 張不同實驗室的咖啡因譜：cosine 最高 {others.cosine.max():.3f}，中位數 {others.cosine.median():.3f}，最低 {others.cosine.min():.3f}")
best = others.loc[others.cosine.idxmax()]
plot_mirror(spec_a, get_spec(best), title=f"Caffeine: {row.contributor} (CE {row.collision_energy}) vs {best.contributor} (CE {best.collision_energy})")

<details><summary>💡 練習 5 提示（先自己試，卡住再打開）</summary>

要找「貢獻者和 `row` 不同」的列：`caf.contributor != row.contributor`；空格要填的是第一筆咖啡因紀錄的變數名稱。

<details><summary>參考答案</summary>

```python
others = caf[caf.contributor != row.contributor]
```
</details>
</details>

In [ ]:
# 5-2 對照：兩個「分子式相同、結構不同」的化合物（同分異構物）
iso1 = lib[lib.name.str.lower() == "sulfisoxazole"].iloc[0]
iso2 = lib[lib.name.str.lower() == "sulfamoxole"].iloc[0]
s1, s2 = get_spec(iso1), get_spec(iso2)
sc = sim.pair(s1, s2)
print(f"{iso1['name']}（{iso1.formula}） vs {iso2['name']}（{iso2.formula}）：cosine = {float(sc['score']):.3f}，對上的峰數 = {int(sc['matches'])}")
plot_mirror(s1, s2, title="Isomers: sulfisoxazole vs sulfamoxole")

**想一想**
1. 同一個化合物在不同實驗室得到的譜，cosine 都很高嗎？分數低的那些，碰撞能量和儀器有什麼不同？（碰撞能量很低時幾乎沒碎，高時碎成小碎片，譜長得差很多。）
2. 兩個同分異構物的 cosine 比同一個化合物低，但高不高？只看分數你敢說它們是不同的化合物嗎？（講課裡也有這一對，分數不同是因為挑的是另一筆紀錄，碰撞能量不一樣。）
3. 鏡像圖上「沒有對上」的峰代表什麼？它們該不該扣分？

---
# 延伸（課後／有餘力）

## E1. 同位素比例相似度：哪一個分子式最像？

**為什麼**：Part 3 的理論分布可以拿來和**實測**的 M+1、M+2 比。MS-DIAL 的同位素比例相似度（官方 FAQ 的寫法）：對每個同位素峰 i，`r_i = I(M+i) / I(M)`，相似度是 **1 − |r（實測，i）− r（理論，i）|**，分數越接近 1 越像。（這裡把 i＝1、2、3 的分數取平均；MS-DIAL 2015 論文如何合併各 i 略有不同，請看它的文件。）

**資料**：MassBank 只有 MS/MS、沒有 MS1，所以這裡的「實測值」是用理論分布加上 ±5% 以內的隨機誤差**模擬**出來的。候選分子式取庫裡 m/z 與 feature 接近（±0.2 Da）的化合物。

In [ ]:
def s_ratio(obs, theo):
    """obs、theo：M 設為 100 的強度陣列；回傳 i = 1..3 的相似度平均。"""
    r_obs, r_theo = np.asarray(obs[1:4]) / obs[0], np.asarray(theo[1:4]) / theo[0]
    return float(np.mean(1 - np.abs(r_obs - r_theo)))

rng = np.random.default_rng(1)
true_pat = isotope_pattern("C14H22ClN3O2")
obs_pat = true_pat * (1 + rng.uniform(-0.05, 0.05, size=true_pat.size)); obs_pat = obs_pat / obs_pat[0] * 100
print("模擬的實測：", np.round(obs_pat[:4], 1))

near = candidates_da(300.1473, 0.2).drop_duplicates("formula")
res = pd.DataFrame({"name": near.name, "formula": near.formula,
                    "S_ratio": [s_ratio(obs_pat, isotope_pattern(f)) for f in near.formula]}).sort_values("S_ratio", ascending=False)
display(res.round(3).head(8))

**想一想**
1. 排第一名和第二名的分子式各是哪一個？它們的分數差多少？第二名也含 Cl，這代表什麼？
2. 實測誤差如果從 5% 變成 15%，排名會不會變？
3. 同位素比例相似度只用到 MS1。要確定一個分子式是對的，還需要什麼？

---
**下一步**：實作二會用同一份資料做**譜庫比對**：把一個實驗室的譜當成「未知」，去其他實驗室的譜庫裡找最像的，看哪些命中是對的、哪些是高分的假命中。